In [ ]:
!pip install pycm umap-learn
import matplotlib.pyplot as plt
import numpy
import pandas
import pycm
import scipy.optimize
import scipy.stats
import seaborn
import sklearn.cluster
import sklearn.datasets
import sklearn.decomposition
import sklearn.manifold
import sklearn.metrics
import tensorflow
import tensorboard
import torch
import torch.utils.tensorboard
import umap


tensorflow.io.gfile = tensorboard.compat.tensorflow_stub.io.gfile

# Clustering with linear and non-linear projections

## Data

From sklearn: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html

In [ ]:
digits = sklearn.datasets.load_digits()
print(digits.data.shape)
print(digits.target.shape)
seaborn.countplot(x=digits.target)

In [ ]:
# 5x5 subplot grid creation. figsize is used to have a descent image size
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# Select 25 index at random, without replacement (we don't want to display the same image twice)
random_indexes = numpy.random.choice(digits.data.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    image = digits.data[img_index]
    label = digits.target[img_index]

    ax[i, j].imshow(image.reshape([8,8]), cmap='gray_r')
    ax[i, j].set_title(f"Exemple {img_index} ({label})")
    ax[i, j].axis('off')

## Training

Define the function `train` which takes a feature matrix (`X`) as a parameter and returns a 10-cluster KMeans model trained using the `fit` method of an object instance [sklearn.cluster.KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html?highlight=kmeans#sklearn.cluster.KMeans).

In [ ]:
# Your code here

### Solution

In [ ]:
def train(data: numpy.ndarray) -> sklearn.cluster.KMeans:
  return sklearn.cluster.KMeans(n_clusters=10).fit(data)

kmeans = train(digits.data)

## Using the learned model to label each point

Use the `predict` method to label each point in the dataset.

In [ ]:
# Your code here

### Solution

In [ ]:
clusters = kmeans.predict(digits.data)

# One can also directly use labels_ to retrieve clusters from the training data
clusters = kmeans.labels_

print(clusters[:10])

## Display

Display the images corresponding to each cluster centre using the [`plt.imshow`](https://matplotlib.org/stable/plot_types/arrays/imshow.html) method .

The centre of a cluster is the centroid calculated for each cluster and stored in `kmeans.cluster_centers_`.

In [ ]:
plt.imshow([[1] * 7,
            [0] * 7,
            [0, 1, 0, 0, 0, 1, 0],
            [0, 0, 0, 1, 0, 0, 0],
            [0, 1, 0, 0, 0, 1, 0],
            [0, 0, 1, 1, 1, 0, 0],
            [0] * 7],
          cmap=plt.cm.binary)
plt.show()

# Your code here

### Solution

In [ ]:
fig, ax = plt.subplots(1, 10, figsize=(8, 3))

centers = kmeans.cluster_centers_.reshape(-1, 8, 8)

for axi, center in zip(ax, centers):
    axi.set(xticks=[], yticks=[])
    axi.imshow(center, cmap=plt.cm.binary)


## Assigning the majority class of each cluster

The function [`scipy.stats.mode`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mode.html) is an efficient way to calculate the mode of each cluster.

Define a `predict` function that takes as arguments a KMeans model and a feature matrix and returns the mode (in targets) of each point's cluster.

We have also seen that it is possible to use the Hungarian algorithm to calculate the assignment of clusters to labels. Use [`scipy.optimize.linear_sum_assignment`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linear_sum_assignment.html#scipy.optimize.linear_sum_assignment) and [`sklearn.metrics.cluster.contingency_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.cluster.contingency_matrix.html) to deploy this method.

In [ ]:
# Your code here

### Solution

In [ ]:
def predict_mode(model: sklearn.cluster.KMeans) -> numpy.ndarray:
  clusters = model.labels_
  predictions = numpy.zeros_like(clusters)
  for i in range(10):
      mask = (clusters == i)
      predictions[mask] = scipy.stats.mode(digits.target[mask])[0]
  return predictions


# Same result by applying the pattern "split, apply, combine"
def predict_mode2(model: sklearn.cluster.KMeans) -> numpy.ndarray:
  return (pandas.Series(digits.target)
                .groupby(model.labels_)
                .transform(lambda x: x.mode()[0])
                .values)

# An optimal solution using a famous assignment problem algorithm
def predict_hungarian(model: sklearn.cluster.KMeans) -> numpy.ndarray:
  clusters = model.labels_
  contmat = sklearn.metrics.cluster.contingency_matrix(digits.target, clusters)
  row_inds, col_inds = scipy.optimize.linear_sum_assignment(contmat, True)
  return col_inds.argsort()[clusters]


predict = predict_hungarian


predictions = predict(kmeans)

## Evaluation

For given predictions, calculate the accuracy using the targets and the [`accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html) function  of sklearn.

Also display the confusion matrix between labels and clusters. Instead of using sklearn's confusion matrix, use the [`pycm`](https://www.pycm.ir/doc/index.html) package which is more advanced.

Finally, calculate the [silhouette](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html) scores and the [adjusted Rand index](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html) of the resulting clusters.

In [ ]:
# Your code here

### Solution

In [ ]:
def evaluate(data: numpy.ndarray,
             predictions: numpy.ndarray,
             clusters: numpy.ndarray
             ) -> None:
  accuracy = sklearn.metrics.accuracy_score(digits.target, predictions)
  silhouette = sklearn.metrics.silhouette_score(data, clusters)
  rand_score = sklearn.metrics.cluster.adjusted_rand_score(digits.target,
                                                           clusters)
  print(f"Accuracy: {accuracy:.2f}, silhouette: {silhouette:.2f}, "
        f"rand_score: {rand_score:.2f}")
  mat = pycm.ConfusionMatrix(actual_vector=digits.target,
                             predict_vector=predictions)
  mat.plot()
  plt.show()
  if data.shape[1] == 2:
    _, (ax_left, ax_middle, ax_right) = plt.subplots(1, 3, figsize=(12, 4))
    ax_left.scatter(data[:, 0], data[:, 1], c=digits.target, alpha=0.7)
    ax_left.set_title("Classes")
    ax_left.axis("off")
    ax_middle.scatter(data[:, 0], data[:, 1], c=predictions, alpha=0.7)
    ax_middle.set_title("Predictions")
    ax_middle.axis("off")
    ax_right.scatter(data[:, 0], data[:, 1], c=clusters, alpha=0.7)
    ax_right.set_title("Clusters")
    ax_right.axis("off")
    plt.show()


evaluate(digits.data, predictions, kmeans.labels_)

## Pipeline

Combine the `train`, `predict` and `evaluate` functions to define a `pipeline` function which takes a feature matrix as a parameter and displays the evaluation of a KMeans trained on it.

Apply this pipeline :

- to the original data
- to PCA projected data
- to UMAP projected data
- to t-SNE projected data

In [ ]:
# Your code here

### Solution

In [ ]:
def pipeline(data):
  model = train(data)
  clusters = model.labels_
  predictions = predict(train(data))
  evaluate(data, predictions, clusters)


print("Original data")
pipeline(digits.data)
print()

print("PCA")
pca = sklearn.decomposition.PCA(n_components=0.8)
pca.fit(digits.data)
print("Explained variance:", pca.explained_variance_ratio_)
print("Cumulated Explained variance:", numpy.cumsum(pca.explained_variance_ratio_))
pipeline(pca.transform(digits.data))
print()

print("UMAP")
pipeline(umap.UMAP().fit_transform(digits.data))

print("t-SNE")
pipeline(sklearn.manifold.TSNE().fit_transform(digits.data))

## Display with TensorBoard Projector

The TensorBoard tool is a marvel for visualizing training data of all types.

Here we will see how to use it to visualise each point by its original image and target.

Once the tensorboard is launched, select "Projector" from the drop-down menu at the top right. You will then have a representation of the images in a 3 dimensional space.

In order to facilitate the reading, you can select "color by label" in the menu on the left.

**Warning**: a known problem on Tensorboard executed in collaboratory can make the execution of T-SNE and UMAP incorrect. The problem is that the set of factors leading to this result is difficult to identify... If you see a ball-shaped cloud that doesn't burst over the iterations, ask to see a correct execution.

In [ ]:
!rm -rf runs

vectors = numpy.array(digits.data)
metadata = digits.target  # labels
images = torch.LongTensor(digits.data.reshape(-1, 1, 8, 8))
writer = torch.utils.tensorboard.SummaryWriter()
writer.add_embedding(vectors, metadata, label_img=images)
writer.close()
%reload_ext tensorboard
%tensorboard --logdir=runs